## Native FDM solver for rollback version

In [1]:
"""
FDM baseline -- ORIGINAL CONVENTION, matching the notebook in new.md /
Technical_Logs.md exactly (the "first model" -- eta = phi - E_eq, rate
clamped at 50.0, phi never touched by the reaction term).

This is deliberately a DIFFERENT file from fdm_v21 (grounded-steel,
gamma-coupled, unclamped). Do not mix .npy outputs between the two --
they are ground truth for two different physical systems. This file
produces the ground truth for the *original* strong-form PINN checkpoint
(manual_physics_audit showing C=0.736391, phi=-0.894584, dC/dy=-0.037).

Physics, exactly matching interior_pde / butler_volmer_bc in the notebook:
    eta = phi - E_eq
    exponent = clamp(k_const * eta, -10, 10)
    reaction_rate = clamp(Da * C * exp(exponent), max=50.0)
    res_bv: dC/dy_norm + reaction_rate = 0
    res_laplace: d2phi/dx2 + d2phi/dy2 = 0   (phi has NO reaction coupling)

Because phi's only boundary condition is phi=0 at y=0 (Dirichlet) with
zero-flux (Neumann) on the other three sides, and no source term anywhere,
phi's unique self-consistent solution under this equation set is the
constant field phi=0 -- this is expected and matches what was established
analytically and numerically earlier in this project. It also means the
PINN's phi varying with x/t (as seen in every audit of this checkpoint)
is not explained by these governing equations -- that's the substance of
audit-log item 4, not a bug in this solver.
"""

import numpy as np
import scipy.sparse as sp
from scipy.sparse.linalg import spsolve


def build_index(Nx, Ny):
    return lambda i, j: i * Ny + j


def stretched_y_grid(Ny, k=4.0):
    xi = np.linspace(0.0, 1.0, Ny)
    y = np.tanh(k * xi) / np.tanh(k)
    y[0] = 0.0
    y[-1] = 1.0
    return y


def y_second_deriv_coeffs(y):
    Ny = len(y)
    coeffs = np.zeros((Ny, 3))
    for j in range(1, Ny - 1):
        h1 = y[j] - y[j - 1]
        h2 = y[j + 1] - y[j]
        coeffs[j, 0] = 2.0 / (h1 * (h1 + h2))
        coeffs[j, 1] = -2.0 / (h1 * h2)
        coeffs[j, 2] = 2.0 / (h2 * (h1 + h2))
    return coeffs


def y_boundary_deriv_coeffs(y):
    Ny = len(y)
    h1 = y[Ny - 1] - y[Ny - 2]
    h2 = y[Ny - 2] - y[Ny - 3]
    H = h1 + h2
    b_j = (H + h1) / (h1 * H)
    b_jm1 = -H / (h1 * h2)
    b_jm2 = h1 / (H * h2)
    return b_j, b_jm1, b_jm2


def solve_phi_field(Nx, Ny, y, dx):
    """
    Static (time-independent) Laplace solve for phi -- decoupled from C,
    exactly as in the original notebook's interior_pde (no reaction term
    touches phi anywhere). Closed with Neumann/zero-flux on y=1, x=0, x=1,
    since the notebook never specifies a BC there either (same closure
    used throughout this project's FDM baselines for this reason).
    """
    N = Nx * Ny
    idx = build_index(Nx, Ny)
    A = sp.lil_matrix((N, N))
    b = np.zeros(N)
    y2c = y_second_deriv_coeffs(y)
    b_j, b_jm1, b_jm2 = y_boundary_deriv_coeffs(y)

    for i in range(Nx):
        for j in range(Ny):
            k_ = idx(i, j)
            if j == 0:
                A[k_, k_] = 1.0
                b[k_] = 0.0
            elif j == Ny - 1:
                A[k_, k_] = b_j
                A[k_, idx(i, j - 1)] = b_jm1
                A[k_, idx(i, j - 2)] = b_jm2
                b[k_] = 0.0
            elif i == 0:
                A[k_, k_] = 3.0
                A[k_, idx(1, j)] = -4.0
                A[k_, idx(2, j)] = 1.0
                b[k_] = 0.0
            elif i == Nx - 1:
                A[k_, k_] = 3.0
                A[k_, idx(Nx - 2, j)] = -4.0
                A[k_, idx(Nx - 3, j)] = 1.0
                b[k_] = 0.0
            else:
                cL, cC, cR = y2c[j]
                A[k_, k_] = -2.0 / dx**2 + cC
                A[k_, idx(i - 1, j)] = 1.0 / dx**2
                A[k_, idx(i + 1, j)] = 1.0 / dx**2
                A[k_, idx(i, j - 1)] = cL
                A[k_, idx(i, j + 1)] = cR
                b[k_] = 0.0

    return spsolve(A.tocsr(), b)


def bv_reaction_rate_and_deriv(C, phi, Da, params):
    """Exactly matches the notebook's butler_volmer_bc: eta=phi-E_eq, rate capped at 50."""
    if Da == 0.0:
        return np.zeros_like(C), np.zeros_like(C)

    alpha, F, R, T, E_eq = params["alpha"], params["F"], params["R"], params["T"], params["E_eq"]
    k_const = alpha * F / (R * T)

    eta = phi - E_eq  # ORIGINAL convention -- matches notebook exactly
    raw_exponent = k_const * eta
    exponent = np.clip(raw_exponent, -10.0, 10.0)
    exp_clamp_active = (raw_exponent < -10.0) | (raw_exponent > 10.0)

    exp_term = np.exp(exponent)
    raw_rate = Da * C * exp_term
    rate = np.minimum(raw_rate, 50.0)  # ORIGINAL clamp -- matches notebook exactly
    rate_clamp_active = raw_rate > 50.0

    active = (~exp_clamp_active) & (~rate_clamp_active)
    d_rate_dC = np.where(active, Da * exp_term, 0.0)
    return rate, d_rate_dC


def solve_fdm_baseline(Nx, Ny, Nt, dt, L_meters, params, y_stretch_k=6.0, Da_override=None, verbose=False):
    dx = 1.0 / (Nx - 1)
    y = stretched_y_grid(Ny, k=y_stretch_k)
    idx = build_index(Nx, Ny)
    N = Nx * Ny

    Fo = params["D_global"] * params["t_max"] / (L_meters ** 2)
    Da = params["k_rate"] * L_meters / params["D_global"] if Da_override is None else Da_override

    y2c = y_second_deriv_coeffs(y)
    b_j, b_jm1, b_jm2 = y_boundary_deriv_coeffs(y)

    # phi is static and fully decoupled from C in this equation set -- solve once.
    phi_flat = solve_phi_field(Nx, Ny, y, dx)

    C_history = np.zeros((Nt, Nx, Ny))
    C_flat_prev = np.zeros(N)
    newton_iters_log = []
    max_rate_log = []

    for n in range(1, Nt):
        C_flat = C_flat_prev.copy()
        n_newton = 0

        for it in range(30):
            n_newton = it + 1
            A = sp.lil_matrix((N, N))
            R_vec = np.zeros(N)
            max_rate_this_iter = 0.0

            for i in range(Nx):
                for j in range(Ny):
                    k_ = idx(i, j)

                    if j == 0:
                        A[k_, k_] = 1.0
                        R_vec[k_] = C_flat[k_] - 1.0
                        continue

                    if j == Ny - 1:
                        Ck, phik = C_flat[k_], phi_flat[k_]
                        rate_k, drate_dC_k = bv_reaction_rate_and_deriv(
                            np.array([Ck]), np.array([phik]), Da, params
                        )
                        rate_k, drate_dC_k = rate_k[0], drate_dC_k[0]
                        max_rate_this_iter = max(max_rate_this_iter, abs(rate_k))

                        dCdy = (b_j * C_flat[k_] + b_jm1 * C_flat[idx(i, j - 1)]
                                + b_jm2 * C_flat[idx(i, j - 2)])
                        R_vec[k_] = dCdy + rate_k

                        A[k_, k_] = b_j + drate_dC_k
                        A[k_, idx(i, j - 1)] = b_jm1
                        A[k_, idx(i, j - 2)] = b_jm2
                        continue

                    if i == 0:
                        A[k_, k_] = 3.0
                        A[k_, idx(1, j)] = -4.0
                        A[k_, idx(2, j)] = 1.0
                        R_vec[k_] = 3 * C_flat[k_] - 4 * C_flat[idx(1, j)] + C_flat[idx(2, j)]
                        continue

                    if i == Nx - 1:
                        A[k_, k_] = 3.0
                        A[k_, idx(Nx - 2, j)] = -4.0
                        A[k_, idx(Nx - 3, j)] = 1.0
                        R_vec[k_] = 3 * C_flat[k_] - 4 * C_flat[idx(Nx - 2, j)] + C_flat[idx(Nx - 3, j)]
                        continue

                    cL, cC, cR = y2c[j]
                    coef_center = 1.0 / dt + 2 * Fo / dx**2 - Fo * cC
                    A[k_, k_] = coef_center
                    A[k_, idx(i - 1, j)] = -Fo / dx**2
                    A[k_, idx(i + 1, j)] = -Fo / dx**2
                    A[k_, idx(i, j - 1)] = -Fo * cL
                    A[k_, idx(i, j + 1)] = -Fo * cR

                    R_vec[k_] = (coef_center * C_flat[k_]
                                 - Fo / dx**2 * C_flat[idx(i - 1, j)]
                                 - Fo / dx**2 * C_flat[idx(i + 1, j)]
                                 - Fo * cL * C_flat[idx(i, j - 1)]
                                 - Fo * cR * C_flat[idx(i, j + 1)]
                                 - C_flat_prev[k_] / dt)

            delta = spsolve(A.tocsr(), -R_vec)
            C_flat = C_flat + delta
            if np.max(np.abs(delta)) < 1e-10:
                break

        newton_iters_log.append(n_newton)
        max_rate_log.append(max_rate_this_iter)
        if verbose and (n % max(1, Nt // 10) == 0 or n == 1):
            print(f"  t-step {n}/{Nt-1} | newton_iters={n_newton} | max|rate| at wall={max_rate_this_iter:.4e} | max|C|={np.max(np.abs(C_flat)):.4e}")

        C_history[n] = C_flat.reshape(Nx, Ny)
        C_flat_prev = C_flat.copy()

    phi_field = phi_flat.reshape(Nx, Ny)
    return C_history, phi_field, y, newton_iters_log, max_rate_log


def analytic_slab_solution(y, t, Fo, n_terms=200):
    u = np.zeros_like(y)
    for n in range(n_terms):
        lam = (2 * n + 1) * np.pi / 2.0
        u += (2.0 / lam) * np.sin(lam * y) * np.exp(-lam**2 * Fo * t)
    return 1.0 - u


def run_validation():
    print("=" * 60)
    print("VALIDATION: Da=0 (pure diffusion) vs analytic slab solution")
    print("=" * 60)

    params = {
        "D_global": 1e-9, "k_rate": 1e-5, "alpha": 0.5, "F": 96485.0,
        "R": 8.314, "T": 298.15, "E_eq": -0.44, "t_max": 31536000.0,
    }
    WT_INCHES = 0.5
    L_METERS = WT_INCHES * 0.0254
    Fo = params["D_global"] * params["t_max"] / (L_METERS ** 2)

    Nx, Ny, Nt = 5, 61, 30
    dt = 1.0 / (Nt - 1)

    C_hist, phi, y, _, _ = solve_fdm_baseline(
        Nx, Ny, Nt, dt, L_METERS, params, y_stretch_k=4.0, Da_override=0.0
    )

    t_check = [0.05, 0.1, 0.2]
    max_errs = []
    for t_val in t_check:
        n_idx = int(round(t_val / dt))
        C_numeric = C_hist[n_idx, Nx // 2, :]
        C_analytic = analytic_slab_solution(y, n_idx * dt, Fo)
        err = np.abs(C_numeric - C_analytic)
        rel_l2 = np.linalg.norm(err) / (np.linalg.norm(C_analytic) + 1e-12)
        max_errs.append(rel_l2)
        print(f"t={n_idx*dt:.4f} | max abs err: {err.max():.2e} | relative L2 err: {rel_l2:.2e}")

    print(f"\nWorst-case relative L2 error across checked timesteps: {max(max_errs):.2e}")
    return max(max_errs)


if __name__ == "__main__":
    worst_err = run_validation()

    print()
    print("=" * 60)
    print("NONLINEAR CASE: WT=0.5in, ORIGINAL convention (eta=phi-E_eq, clamped rate)")
    print("=" * 60)

    params = {
        "D_global": 1e-9, "k_rate": 1e-5, "alpha": 0.5, "F": 96485.0,
        "R": 8.314, "T": 298.15, "E_eq": -0.44, "t_max": 31536000.0,
    }
    WT_INCHES = 0.5
    L_METERS = WT_INCHES * 0.0254

    Nx, Ny, Nt = 21, 61, 1000
    dt = 1.0 / (Nt - 1)

    C_history, phi_field, y_grid, newton_log, rate_log = solve_fdm_baseline(
        Nx, Ny, Nt, dt, L_METERS, params, y_stretch_k=6.0, verbose=True
    )

    np.save("fdm_C_history_original.npy", C_history)
    np.save("fdm_phi_field_original.npy", phi_field)
    np.save("fdm_y_grid_original.npy", y_grid)

    print("\nMax Newton iterations used at any timestep:", max(newton_log))
    print("Max |rate| ever seen at wall:", max(rate_log))
    print("Any NaN in C or phi?", np.isnan(C_history).any(), np.isnan(phi_field).any())
    print("phi field (should be ~0 everywhere -- decoupled Laplace, no source):")
    print("  max|phi| =", np.max(np.abs(phi_field)))

    print("\nC at final timestep, mid-x, near-wall:")
    for j in range(Ny - 8, Ny):
        print(f"  y={y_grid[j]:.6f} | C={C_history[-1, Nx // 2, j]:.8e}")

VALIDATION: Da=0 (pure diffusion) vs analytic slab solution
t=0.0345 | max abs err: 6.98e-02 | relative L2 err: 6.46e-02
t=0.1034 | max abs err: 2.32e-04 | relative L2 err: 2.13e-04
t=0.2069 | max abs err: 4.25e-08 | relative L2 err: 3.90e-08

Worst-case relative L2 error across checked timesteps: 6.46e-02

NONLINEAR CASE: WT=0.5in, ORIGINAL convention (eta=phi-E_eq, clamped rate)
  t-step 1/999 | newton_iters=2 | max|rate| at wall=4.7958e-01 | max|C|=1.0000e+00
  t-step 100/999 | newton_iters=1 | max|rate| at wall=1.0000e+00 | max|C|=1.0000e+00
  t-step 200/999 | newton_iters=1 | max|rate| at wall=1.0000e+00 | max|C|=1.0000e+00
  t-step 300/999 | newton_iters=1 | max|rate| at wall=1.0000e+00 | max|C|=1.0000e+00
  t-step 400/999 | newton_iters=1 | max|rate| at wall=1.0000e+00 | max|C|=1.0000e+00
  t-step 500/999 | newton_iters=1 | max|rate| at wall=1.0000e+00 | max|C|=1.0000e+00
  t-step 600/999 | newton_iters=1 | max|rate| at wall=1.0000e+00 | max|C|=1.0000e+00
  t-step 700/999 | newt